# VoxCPM2 deepfake generation (Kaggle) -- Swahili Common Voice

**Kaggle port of the VoxCPM2 variant** (OpenBMB VoxCPM2, Apache-2.0, 2B params, 30 languages incl. Swahili,
48 kHz output). Same design as the finished Chatterbox run: reference clips + manifests come from Google Drive
via rclone (Kaggle Secret `RCLONE_CONF`), generation runs in checkpoints of `CHECKPOINT_SIZE`, and everything
is pushed back to `swahili-deepfake-dataset/data/generated/voxcpm/` after each checkpoint.

**`SMOKE_ONLY = True` (default for the first run)** runs only the 12-clip smoke test, pushes those clips to
`generated_smoke/voxcpm/` on Drive for listening, and skips the full generation. Set it to `False` after the
smoke audio sounds right.


## 0. Configuration

In [ ]:
# ---- Source of the real reference audio ----
#   "drive" : pull the already-preprocessed clips from Google Drive (default;
#             fast, no multi-GB download). Requires the RCLONE_CONF secret.
#   "mdc"   : rebuild everything from the Mozilla Data Collective archive
#             (Section 4). Only needed if the Drive copy is gone.
DATA_SOURCE = "drive"

# rclone remote name -- must match the [name] header inside your RCLONE_CONF.
RCLONE_REMOTE = "gdrive"
# Folder scripts/sync_drive.py backed up to, relative to the remote root (My Drive).
DRIVE_DATA_DIR = "swahili-deepfake-dataset/data"

# Push generated deepfakes + manifests back to Drive after each checkpoint.
# Requires the rclone remote to have write scope (not drive.readonly).
PUSH_RESULTS_TO_DRIVE = True

# VoxCPM2
DEVICE = "cuda"
SMOKE_ONLY = True          # True: 12-clip smoke test only (pushed to Drive/generated_smoke)
SMOKE_N = 12
CHECKPOINT_SIZE = 500        # generate in batches of this many, sync between

REPO_URL = "https://github.com/regak/Swahili-Deepfake-dataset.git"
REPO_REF = "main"   # branch/tag that contains the chatterbox + voxcpm generators and the tmp-file fix
REPO_DIR = "/kaggle/working/Swahili-Deepfake-dataset"
SCRATCH = "/kaggle/temp"     # not persisted; fine for smoke-test artefacts

RUN_MDC = (DATA_SOURCE == "mdc")
NEED_RCLONE = (DATA_SOURCE == "drive") or PUSH_RESULTS_TO_DRIVE

import os
os.makedirs(SCRATCH, exist_ok=True)
print(f"DATA_SOURCE={DATA_SOURCE}  RUN_MDC={RUN_MDC}  NEED_RCLONE={NEED_RCLONE}")

## 1. Clone the pipeline repo and install dependencies

In [ ]:
!git clone --depth 1 --branch {REPO_REF} {REPO_URL} {REPO_DIR}

import os
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

!pip install -q -r requirements.txt
!pip install -q voxcpm soundfile

# Fail fast if the install broke the numpy/numba/torch stack (same trap as the Chatterbox run).
import subprocess, sys
t = subprocess.run([sys.executable, '-c', 'import numpy, numba, torch, soundfile; from voxcpm import VoxCPM; print("numpy", numpy.__version__, "torch", torch.__version__)'], capture_output=True, text=True)
print(t.stdout, t.stderr[-1500:])
assert t.returncode == 0, "voxcpm / numpy / numba / torch import check failed -- see output above"


In [ ]:
import csv, sys
# Python's csv module defaults to a 128 KB field cap; the pipeline's TSVs can
# legitimately exceed it. Same bump the upstream scripts do.
try:
    csv.field_size_limit(sys.maxsize)
except OverflowError:
    csv.field_size_limit(2**31 - 1)

## 2. Install rclone and load the Drive remote from Kaggle Secrets

Kaggle has no `google.colab.drive`, so this uses `rclone` against the same
`MyDrive/swahili-deepfake-dataset/data/` folder `scripts/sync_drive.py` writes
to on Colab. The static binary needs no `apt`; the remote definition comes
from the `RCLONE_CONF` secret so no token is ever written into a cell.

In [ ]:
if NEED_RCLONE:
    !curl -sL https://downloads.rclone.org/rclone-current-linux-amd64.zip -o {SCRATCH}/rclone.zip
    !cd {SCRATCH} && unzip -oq rclone.zip && cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod 755 /usr/local/bin/rclone
    !rclone version | head -n 1

    from pathlib import Path
    from kaggle_secrets import UserSecretsClient

    conf_dir = Path.home() / ".config" / "rclone"
    conf_dir.mkdir(parents=True, exist_ok=True)
    import re
    _raw = UserSecretsClient().get_secret("RCLONE_CONF")
    # Kaggle's secret box flattens the config onto one line; rebuild one setting per line.
    _flat = " ".join(_raw.replace(chr(0xFEFF), "").split())
    _known = "type|client_id|client_secret|scope|token|team_drive|root_folder_id|service_account_file|service_account_credentials|auth_url|token_url"
    _parts = re.split("[ ](?=(?:" + _known + ")[ ]*=)", _flat)
    _conf = chr(10).join(_parts).strip() + chr(10)
    (conf_dir / "rclone.conf").write_text(_conf)
    # Diagnostics: setting NAMES only -- never print values (they contain the login token).
    _has_remote = _parts[0].strip() == "[" + RCLONE_REMOTE + "]"
    _names = [p.split("=")[0].strip() for p in _parts[1:]]
    print("config rebuilt: remote header ok =", _has_remote, "| settings found =", _names)
    assert _has_remote, "RCLONE_CONF does not start with the [gdrive] header"
    assert any(p.startswith("token") and len(p) > 60 for p in _parts), "RCLONE_CONF has no OAuth token"
    print("Wrote", conf_dir / "rclone.conf")

    # Sanity check: list the backed-up subfolders.
    !rclone lsd "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}"
else:
    print("Skipping rclone (DATA_SOURCE='mdc' and PUSH_RESULTS_TO_DRIVE=False).")

## 3. Restore preprocessed reference audio + manifests from Drive

`data/processed/` holds the 16 kHz mono WAV reference clips (named
`<utterance-id-stem>.wav`) from the previous run's `preprocess_audio.py` step.
`data/generated/` (if present) is restored too, so an interrupted run resumes
instead of regenerating.

In [ ]:
from pathlib import Path

if DATA_SOURCE == "drive":
    Path("data/processed").mkdir(parents=True, exist_ok=True)
    Path("data/manifests").mkdir(parents=True, exist_ok=True)

    !rclone copy "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/processed"  data/processed  --transfers 32 --checkers 32 --fast-list --stats 15s
    !rclone copy "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/manifests" data/manifests --fast-list
    !rclone copy "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/generated/voxcpm" data/generated/voxcpm --transfers 32 --checkers 32 --fast-list --stats 15s || true

    print("\nRestored:")
    !find data/processed -type f | wc -l
    !ls data/manifests
else:
    print("DATA_SOURCE='mdc' -> skipping Drive restore; run Section 4 instead.")

## 5. Resolve the working subset

Runs in either mode. Intersects `selected_subset.tsv` with the reference clips
actually on disk and writes `selected_subset.present.tsv` — generation below
uses only that, so the deepfake count equals the real-clip count 1:1 (no
attempts against missing audio, no silent divergence).

In [ ]:
import csv
from pathlib import Path

SUBSET_TSV = "data/manifests/selected_subset.tsv"
PRESENT_SUBSET_TSV = "data/manifests/selected_subset.present.tsv"
PROCESSED_DIR = Path("data/processed")
AUDIO_EXT = ".wav"

processed_stems = {p.stem for p in PROCESSED_DIR.glob(f"*{AUDIO_EXT}")}
print(f"Reference clips in data/processed: {len(processed_stems)}")

with open(SUBSET_TSV, newline="", encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f, delimiter="\t"))
present_rows = [r for r in all_rows if Path(r["id"]).stem in processed_stems]
print(f"selected_subset.tsv rows: {len(all_rows)}  |  with reference audio on disk: {len(present_rows)}")

with open(PRESENT_SUBSET_TSV, "w", newline="", encoding="utf-8") as fout:
    w = csv.DictWriter(fout, fieldnames=list(all_rows[0].keys()), delimiter="\t")
    w.writeheader()
    w.writerows(present_rows)

N_TARGET = len(present_rows)
print(f"\nWrote {PRESENT_SUBSET_TSV} -> {N_TARGET} VoxCPM2 deepfakes will be generated.")
assert N_TARGET > 0, "No reference audio matched selected_subset.tsv -- check Section 3 / Section 4."

## 7. Sanity checks + baseline (real-only) manifest

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

In [ ]:
# Empty fake manifest (header only) -- lets build_manifest.py run before any
# deepfakes exist, as a real-data sanity check.
!mkdir -p data/manifests
import os
if not os.path.exists("data/manifests/fake_manifest.tsv"):
    with open("data/manifests/fake_manifest.tsv", "w") as f:
        f.write("id\tsource_utterance_id\tspeaker_id\tgenerator\taudio_path\n")

!python scripts/build_manifest.py \
    data/manifests/selected_subset.present.tsv data/manifests/fake_manifest.tsv \
    data/processed --output data/manifests/dataset_manifest.csv \
    --summary data/manifests/split_summary.json

## 8. Smoke test -- run this first

Confirms the generator loads, runs on GPU, and writes clips. 12 clips are pushed to Drive under
`generated_smoke/voxcpm/` so they can be listened to before the full run.

In [ ]:
!mkdir -p {SCRATCH}
!head -n {SMOKE_N + 1} data/manifests/selected_subset.present.tsv > {SCRATCH}/smoke_subset.tsv

import time
_t0 = time.time()
!python scripts/generate_deepfakes.py \
    {SCRATCH}/smoke_subset.tsv data/processed {SCRATCH}/smoke_generated \
    --generators voxcpm \
    --device {DEVICE} \
    --manifest {SCRATCH}/smoke_fake_manifest.tsv \
    --progress-every 1
assert get_ipython().user_ns.get('_exit_code', 0) == 0, 'generation failed -- see traceback above'
print(f"smoke test wall time: {time.time() - _t0:.0f}s for {SMOKE_N} clips (includes model load)")


In [ ]:
import glob, soundfile as sf
files = sorted(glob.glob(f"{SCRATCH}/smoke_generated/voxcpm/*.wav"))
assert files, "smoke test produced no wav files"
for f in files:
    i = sf.info(f)
    print(f.split("/")[-1], f"{i.frames / i.samplerate:.2f}s", i.samplerate, "Hz")

if PUSH_RESULTS_TO_DRIVE:
    !rclone copy {SCRATCH}/smoke_generated/voxcpm "{RCLONE_REMOTE}:swahili-deepfake-dataset/generated_smoke/voxcpm" --fast-list
    print("Smoke clips pushed to Drive: swahili-deepfake-dataset/generated_smoke/voxcpm")

from IPython.display import Audio, display
display(Audio(files[0]))


## 9. Full generation, checkpointed every `CHECKPOINT_SIZE` utterances

Skipped while `SMOKE_ONLY = True`.

In [ ]:
if SMOKE_ONLY:
    print('SMOKE_ONLY=True -> skipping full generation.')
else:
    import math

    with open("data/manifests/selected_subset.present.tsv", newline="", encoding="utf-8") as f:
        header = f.readline()
        lines = f.readlines()

    total = len(lines)
    num_checkpoints = math.ceil(total / CHECKPOINT_SIZE)
    print(f"{total} utterances -> {num_checkpoints} checkpoint(s) of {CHECKPOINT_SIZE}")

    for i in range(1, num_checkpoints + 1):
        cutoff = min(i * CHECKPOINT_SIZE, total)
        prefix_path = f"{SCRATCH}/subset_prefix.tsv"
        with open(prefix_path, "w", encoding="utf-8") as pf:
            pf.write(header)
            pf.writelines(lines[:cutoff])

        print(f"--- Checkpoint {i}/{num_checkpoints}: utterances 1-{cutoff} of {total} ---")
        !python scripts/generate_deepfakes.py {prefix_path} data/processed data/generated \
            --generators voxcpm --device {DEVICE} --progress-every 10 \
            --manifest data/manifests/fake_manifest_voxcpm.tsv
        assert get_ipython().user_ns.get('_exit_code', 0) == 0, 'generation failed -- see traceback above'

        if PUSH_RESULTS_TO_DRIVE:
            print(f"Pushing to Drive after {cutoff} utterances...")
            !rclone copy data/generated/voxcpm "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/generated/voxcpm" --transfers 32 --checkers 32 --fast-list
            !rclone copy data/manifests "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/manifests" --fast-list


In [ ]:
if SMOKE_ONLY:
    print('SMOKE_ONLY=True -> skipping manifest merge.')
else:
    # VoxCPM2 writes its own manifest so earlier generators' rows are never overwritten.
    # Merge f5_tts_sw + chatterbox + voxcpm into one file for build_manifest.py.
    import os
    rows = []
    for p in ('data/manifests/fake_manifest.tsv', 'data/manifests/fake_manifest_chatterbox.tsv', 'data/manifests/fake_manifest_voxcpm.tsv'):
        if os.path.exists(p):
            with open(p, encoding='utf-8') as f:
                lines = f.read().splitlines()
            if not rows:
                rows.append(lines[0])
            rows.extend(l for l in lines[1:] if l.strip())
            print(p, len(lines) - 1, 'rows')
    with open('data/manifests/fake_manifest_combined.tsv', 'w', encoding='utf-8') as f:
        f.write('\n'.join(rows) + '\n')
    print('combined manifest rows:', len(rows) - 1)

    import glob
    n_fake = len(glob.glob("data/generated/voxcpm/*.wav"))
    print(f"real reference clips : {N_TARGET}")
    print(f"voxcpm deepfakes     : {n_fake}")

    !python scripts/build_manifest.py \
        data/manifests/selected_subset.present.tsv data/manifests/fake_manifest_combined.tsv \
        data/processed --output data/manifests/dataset_manifest.csv \
        --summary data/manifests/split_summary.json


## 10. Final sync

In [ ]:
if SMOKE_ONLY:
    print('SMOKE_ONLY=True -> done (smoke clips are on Drive under generated_smoke/voxcpm).')
else:
    if PUSH_RESULTS_TO_DRIVE:
        !rclone copy data/generated/voxcpm "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/generated/voxcpm" --transfers 32 --checkers 32 --fast-list
        !rclone copy data/manifests "{RCLONE_REMOTE}:{DRIVE_DATA_DIR}/manifests" --fast-list
        print("Final push to Drive done.")
